In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import datetime as dt

In [26]:
#optional for visualization
from lonboard import viz, Map, ScatterplotLayer, PolygonLayer
from lonboard.colormap import apply_continuous_cmap

In [42]:
perims = gpd.read_file('/projects/shared-buckets/coffield/2020_feds_large_fires95_buffered.geojson')
perims['start_str'] = perims.startdate.astype(str) #helps during viz
perims['end_str'] = perims.enddate.astype(str)
perims.head()

,fireid,farea,startdate,enddate,name,geometry,start_str,end_str
0,F11445,4493.47180365792,2020-08-17 00:00:00,2020-11-25 00:00:00,AUGUST COMPLEX,"POLYGON ((-123.53943 40.27585, -123.53955 40.2...",2020-08-17 00:00:00,2020-11-25 00:00:00
1,F11440,1894.38848432324,2020-08-17 00:00:00,2020-10-29 12:00:00,LIONSHEAD,"POLYGON ((-122.63453 44.78792, -122.63473 44.7...",2020-08-17 00:00:00,2020-10-29 12:00:00
2,F12649,1768.27870403421,2020-09-05 00:00:00,2020-11-26 00:00:00,CREEK,"POLYGON ((-119.50743 37.38098, -119.50743 37.3...",2020-09-05 00:00:00,2020-11-26 00:00:00
3,F11393,1526.054850204,2020-08-16 12:00:00,2020-09-27 12:00:00,SCU LIGHTNING COMPLEX,"POLYGON ((-121.81677 37.46370, -121.81680 37.4...",2020-08-16 12:00:00,2020-09-27 12:00:00
4,F11527,1400.56092016471,2020-08-18 00:00:00,2020-10-29 12:00:00,NORTH COMPLEX,"POLYGON ((-121.49817 39.64681, -121.49862 39.6...",2020-08-18 00:00:00,2020-10-29 12:00:00


<h3>Spatial clipping to buffered fire perimeters

In [85]:
#read in dets, clip to buffered perimeters
dets = pd.DataFrame()

for month in range(6,12):
    m = f'{month:02d}'
    print('reading month', m)
    
    for sat in ['SNPP', 'NOAA20']:
        d = pd.read_csv(f'/projects/shared-buckets/coffield/viirs/outputs/western_us/2020_{m}_dets_{sat}.csv')
        d = gpd.GeoDataFrame(d, geometry=gpd.GeoSeries.from_xy(d['longitude'], d['latitude']), crs=4326)
        d = d.clip(perims)
        d['satellite'] = sat
        dets = pd.concat([dets, d])

dets = dets.reset_index(drop=True)
dets

reading month 06
reading month 07
reading month 08
reading month 09
reading month 10
reading month 11


,longitude,latitude,fire_mask,confidence,acq_date,acq_time,acq_datetime,j,vza,sza,daynight,i750,j750,frp,frp_old,dist_m13b,geometry,satellite
0,-117.324610,33.329407,2.0,x,2020-06-19,20:42,2020-06-19 20:42:00 +00:00,3614.0,11.0715,15.290000,D,1056.0,1807.0,-1.182707,NaN,2.617139,POINT (-117.32461 33.32941),SNPP
1,-117.336655,33.347225,8.0,n,2020-06-10,08:48,2020-06-10 08:48:00 +00:00,967.0,47.4607,121.520000,N,3138.0,483.0,2.425918,2.425918,NaN,POINT (-117.33665 33.34723),SNPP
2,-117.345110,33.347466,8.0,n,2020-06-10,10:30,2020-06-10 10:30:00 +00:00,6028.0,52.7608,111.229996,N,2216.0,3014.0,3.217252,3.217252,NaN,POINT (-117.34511 33.34747),SNPP
3,-117.340454,33.347492,8.0,n,2020-06-10,08:48,2020-06-10 08:48:00 +00:00,966.0,47.4696,121.520000,N,3138.0,483.0,2.425918,2.425918,NaN,POINT (-117.34045 33.34749),SNPP
4,-117.344450,33.347782,9.0,h,2020-06-10,08:48,2020-06-10 08:48:00 +00:00,965.0,47.4785,121.520000,N,3138.0,482.0,5.945793,5.945793,NaN,POINT (-117.34445 33.34778),SNPP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882885,-122.526380,44.740444,8.0,n,2020-11-12,09:30,2020-11-12 09:30:00 +00:00,1672.0,37.7053,145.870000,N,2663.0,836.0,3.447919,3.447919,NaN,POINT (-122.52638 44.74044),NOAA20
882886,-122.614750,44.768620,8.0,n,2020-11-19,20:24,2020-11-19 20:24:00 +00:00,4572.0,34.9486,64.909996,D,2534.0,2286.0,3.756472,3.756472,NaN,POINT (-122.61475 44.76862),NOAA20
882887,-122.689800,44.812138,8.0,n,2020-11-20,10:24,2020-11-20 10:24:00 +00:00,3915.0,19.1017,140.410000,N,693.0,1957.0,0.568781,0.568781,NaN,POINT (-122.68980 44.81214),NOAA20
882888,-122.428800,45.136887,8.0,n,2020-11-18,20:42,2020-11-18 20:42:00 +00:00,3750.0,14.6997,65.720000,D,2995.0,1875.0,5.285265,5.285265,NaN,POINT (-122.42880 45.13689),NOAA20


In [114]:
#spatial join dets to the perim they are within
joined = dets.sjoin(perims, predicate='within').drop(['farea', 'start_str', 'end_str'], axis=1) 
#note some overlap in Arizona

In [115]:
joined.columns

Index(['longitude', 'latitude', 'fire_mask', 'confidence', 'acq_date',
       'acq_time', 'acq_datetime', 'j', 'vza', 'sza', 'daynight', 'i750',
       'j750', 'frp', 'frp_old', 'dist_m13b', 'geometry', 'satellite',
       'index_right', 'fireid', 'startdate', 'enddate', 'name'],
      dtype='object')

<h3>Temporal clipping to +/- 1 day of fire object activity

In [121]:
joined.acq_datetime = pd.to_datetime(joined.acq_datetime, utc=True)
joined.startdate = pd.to_datetime(joined.startdate, utc=True) #actually day or night local
joined.enddate = pd.to_datetime(joined.enddate, utc=True) #actually day or night local

In [124]:
joined = joined[joined.acq_datetime >= joined.startdate - dt.timedelta(days=1)]
joined = joined[joined.acq_datetime <= joined.enddate + dt.timedelta(days=1.5)] #allow extra hours for UTC time difference
len(joined)

873862

<h3>Visualize final set of detections associated with perims

In [125]:
min_bound = perims.index.min()
max_bound = perims.index.max()
normalized_dets_colors = (joined.index_right - min_bound) / (max_bound - min_bound)
normalized_perm_colors = (perims.index - min_bound) / (max_bound - min_bound)

import palettable
cmap = palettable.colorbrewer.diverging.Spectral_8_r

In [126]:
layer1 = PolygonLayer.from_geopandas(
    perims,
    get_line_width=20,  # width in default units (meters)
    line_width_min_pixels=0.2,  # minimum width when zoomed out
    get_fill_color= apply_continuous_cmap(normalized_perm_colors, cmap), #'#e27018', #orange
    get_line_color=[37, 36, 34],  # dark border color
    opacity = 0.5
)

layer2 = ScatterplotLayer.from_geopandas(joined, stroked=True)
layer2.get_fill_color = apply_continuous_cmap(normalized_dets_colors, cmap)
layer2.get_radius = 200
layer2.radius_units = "meters"
layer2.radius_min_pixels = 2
layer2.opacity = 0.7

m = Map([layer1, layer2], _height=800)
m

Map(layers=[PolygonLayer(get_fill_color=<pyarrow.lib.FixedSizeListArray object at 0x7f7ed5041ba0>
[
  [
    50…

<h3>Export as parquet and csv

In [127]:
joined.columns

Index(['longitude', 'latitude', 'fire_mask', 'confidence', 'acq_date',
       'acq_time', 'acq_datetime', 'j', 'vza', 'sza', 'daynight', 'i750',
       'j750', 'frp', 'frp_old', 'dist_m13b', 'geometry', 'satellite',
       'index_right', 'fireid', 'startdate', 'enddate', 'name'],
      dtype='object')

In [ ]:
joined.to_parquet('/projects/my-public-bucket/viirs/outputs/western_us/joined_dets_2020.parquet', index=False)
joined.to_csv('/projects/my-public-bucket/viirs/outputs/western_us/joined_dets_2020.csv', index=False)